# 16 — Online Bookstore Orders: Three-Table Take-Home (Cleaning + Joins + Gold + Unit Tests)

**Scenario.** Boekenhuis is a small Dutch online bookstore. The operations team exported three files: an **orders** export from the webshop, the **customer** list from the CRM, and the **book catalog**. The two systems were never properly integrated, so IDs don't always match cleanly. The goal: reliable numbers for a management meeting.

This is my first multi-table take-home: the main new muscle is **joins** — picking the join type as a business decision, normalizing keys before joining, anti joins, and proving the join didn't duplicate rows.

| file | grain (claimed) | rows |
|---|---|---|
| `orders_16.csv` | one row = one order | 112 |
| `customers_16.csv` | one row = one customer | 22 |
| `books_16.csv` | one row = one book | 12 |

*Synthetic practice dataset. Pipeline: Bronze (raw as string) → Silver (cleaned + joined) → Gold (business questions) + pytest unit tests.*


## 1. Bronze — read everything as string

I never let Spark guess types on dirty data. Reading as string first means nothing is silently coerced or dropped before I've measured it.

In [ ]:
# Read the raw tables and keep them untouched as the bronze layer
books_raw_p16_df = (
    spark.read.format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/books_16.csv")
)

customers_raw_p16_df = (
    spark.read.format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/customers_16.csv")
)

orders_raw_p16_df = (
    spark.read.format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/orders_16.csv")
)

## 2. Profiling — the same ritual for every table

Even a 12-row lookup table gets the full check (count, grain, nulls, duplicates). Small tables could have thousands of rows tomorrow; the ritual stays the same.

In [ ]:
from pyspark.sql.functions import col

# --- books ---
books_raw_p16_df.printSchema()
print(books_raw_p16_df.count())                                    # 12 rows
print(books_raw_p16_df.select("book_id").count())                  # 12
print(books_raw_p16_df.select("book_id").distinct().count())       # 12 -> grain check passed: one row = one book
print(books_raw_p16_df.filter(col("book_id").isNull()).count())    # 0
print(books_raw_p16_df.filter(col("title").isNull()).count())      # 0
print(books_raw_p16_df.filter(col("category").isNull()).count())   # 0
print(books_raw_p16_df.dropDuplicates().count())                   # 12 -> no duplicate rows

In [ ]:
# --- customers ---
customers_raw_p16_df.printSchema()
print(customers_raw_p16_df.count())                                     # 22 rows
print(customers_raw_p16_df.select("customer_id").count())               # 22
print(customers_raw_p16_df.select("customer_id").distinct().count())    # 22 -> customer_id is unique
print(customers_raw_p16_df.filter(col("customer_id").isNull()).count()) # 0
print(customers_raw_p16_df.filter(col("customer_name").isNull()).count()) # 0
print(customers_raw_p16_df.filter(col("city").isNull()).count())        # 0
print(customers_raw_p16_df.dropDuplicates().count())                    # 22

# The uniqueness of customer_id matters later: it is what guarantees
# that joining orders to customers cannot fan out (duplicate order rows).

In [ ]:
# --- orders ---
orders_raw_p16_df.printSchema()
print(orders_raw_p16_df.count())                                   # 112 rows
print(orders_raw_p16_df.select("order_id").count())                # 112
print(orders_raw_p16_df.select("order_id").distinct().count())     # 109 -> 3 order_ids appear twice
print(orders_raw_p16_df.dropDuplicates().count())                  # 109 -> the 3 extras are full-row copies

# Proof, not assumption: show the duplicated ids and eyeball the rows
orders_raw_p16_df.groupBy("order_id").count().filter(col("count") > 1).display()
orders_raw_p16_df.filter(col("order_id").isin("ORD-5097", "ORD-5085", "ORD-5073")).display()

# The rows are identical in every column, so these are true duplicates (double export).
orders_dedup_p16_df = orders_raw_p16_df.dropDuplicates()
print(orders_dedup_p16_df.count())                                 # 112 -> 109

**Measured:** 112 → **109** rows after full-row dedup. The 3 duplicates (ORD-5073, ORD-5085, ORD-5097) were exact copies — proven by `groupBy(order_id).count() > 1` plus looking at the rows, not assumed.

*Lesson I'm carrying from Project 10: normalize-then-dedup is the safer order (case variants can hide near-duplicates). Here the copies were byte-identical so dedup-first worked, but I'd flip the order next time.*

## 3. Cleaning

### 3.1 Normalize — and the trap of this project

The scenario said *"the two systems were never properly integrated — IDs don't always match cleanly."* My first pass only trimmed `customer_id`. Later, my anti join returned **6** "unknown" customers — but 4 of them (`cu-102`, `cu-106`, `cu-109`, `Cu-113`) were just **lowercase variants** of real CRM customers. String join keys don't raise errors when they don't match; the rows just silently fall out of the join.

Fix: normalize the key to the CRM's canonical format — `trim` + `upper` — on the orders side (customers already carried `CU-xxx`). After that, only **2 real orphans** remained.

In [ ]:
from pyspark.sql.functions import trim, initcap, lower, upper

books_clean_p16_df = books_raw_p16_df.withColumns({
    "book_id":  trim(col("book_id")),
    "title":    trim(initcap(col("title"))),
    "category": trim(lower(col("category")))
})

customers_clean_p16_df = customers_raw_p16_df.withColumns({
    "customer_id":   trim(col("customer_id")),
    "customer_name": trim(initcap(col("customer_name"))),
    "city":          trim(initcap(col("city")))
})

orders_normalized_p16_df = orders_dedup_p16_df.withColumns({
    "order_id":    trim(col("order_id")),
    "customer_id": trim(upper(col("customer_id"))),   # <- the fix: same rule as the CRM side
    "book_id":     trim(col("book_id")),
    "quantity":    trim(col("quantity")),
    "unit_price":  trim(col("unit_price")),
    "order_date":  trim(col("order_date")),
    "status":      trim(lower(col("status")))
})

### 3.2 Placeholders → NULL

Systematic sweep over every column before casting. `"N/A"` and `"ERROR"` are not values — leaving them in would either crash the cast (ANSI mode) or silently corrupt counts.

In [ ]:
from pyspark.sql.functions import when

orders_normalized_p16_df = orders_normalized_p16_df.withColumns({
    c: when(lower(col(c)).isin("error", "n/a", "unknown"), None).otherwise(col(c))
    for c in ["order_id", "customer_id", "book_id", "quantity", "unit_price", "order_date", "status"]
})

# Measured: 3 unit_price placeholders -> NULL, 2 status 'unknown' -> NULL. No other column was hit.

### 3.3 Types — clean first, cast after

- Money is **never** int or float: `decimal(10,2)`. (My first draft had `cast("int")` — that would have truncated every cent.)
- Currency dirt (`€`, comma decimals, stray spaces) is removed **before** the cast, so the cast can't silently produce new nulls.
- Dates arrive in 3 formats → `coalesce` of `try_to_date` attempts, then **prove** 0 rows failed to parse.

In [ ]:
from pyspark.sql.functions import coalesce, try_to_date, replace, lit

orders_typed_p16_df = orders_normalized_p16_df.withColumns({
    "quantity":   col("quantity").cast("int"),
    "unit_price": replace(replace(replace("unit_price", lit("€"), lit("")),
                          lit(","), lit(".")), lit("$"), lit("")).cast("decimal(10,2)"),
    "order_date": coalesce(
        try_to_date("order_date", "yyyy-M-d"),
        try_to_date("order_date", "d/M/yyyy"),
        try_to_date("order_date", "MMM d, yyyy")
    )
})

# Post-cast checks: nothing new broke
print(orders_typed_p16_df.filter(col("order_date").isNull()).count())   # 0 -> every date parsed
print(orders_typed_p16_df.filter(col("unit_price").isNull()).count())   # 3 -> same 3 as before the cast (the cast produced no new nulls)

In [ ]:
from pyspark.sql.functions import max as max_, min as min_

orders_typed_p16_df.select(max_("order_date"), min_("order_date")).display()   # 2025-01 .. 2025-06, plausible
print(orders_typed_p16_df.filter(col("unit_price") < 0).count())   # 0 negative prices
print(orders_typed_p16_df.filter(col("quantity") <= 0).count())    # 0 broken quantities

orders_clean_p16_df = orders_typed_p16_df

## 4. Silver — join the three tables

Cleaning came **before** the join on purpose: dirty keys don't error, they silently fail to match; duplicate rows would multiply through a join.

**Join type is a business decision, not a syntax choice.** Two of the orders belong to customer IDs that don't exist in the CRM (`CU-888`, `CU-999`). Those are real sales — real money. An inner join would silently drop them and understate revenue, so I keep **orders on the left with a left join** and add a flag instead of hiding the problem.

In [ ]:
silver_p16_df = (
    orders_clean_p16_df
    .join(customers_clean_p16_df, on="customer_id", how="left")
    .join(books_clean_p16_df, on="book_id", how="left")
    .withColumn("customer_missing_in_crm", col("customer_name").isNull())
)

# Fan-out proof: a join must add columns, never rows.
print(orders_clean_p16_df.count())   # 109
print(silver_p16_df.count())         # 109 -> no fan-out (guaranteed by customer_id being unique in customers)

# The 2 kept orphans, flagged instead of dropped: EUR 56.85 of revenue stays in the report
silver_p16_df.filter(col("customer_missing_in_crm")).display()

**Grain statement (required by the task):** one row in the silver table = **one order** (each order covers one book title), enriched with customer and book attributes. Proven: 109 rows, 109 distinct `order_id`.

In [ ]:
silver_p16_df.createOrReplaceTempView("orders_p16_silver")
orders_clean_p16_df.createOrReplaceTempView("orders_p16")
customers_clean_p16_df.createOrReplaceTempView("customers_p16")
books_clean_p16_df.createOrReplaceTempView("books_p16")

## 5. Gold questions

### G1 — Customers who never ordered (anti join)

Marketing wants a win-back list. The clean pattern for *"in A but not in B"* is a **left anti join** with the subject on the left: it returns the left-side rows that have no match — no null filter needed.

My memory hook, learned the hard way after putting the wrong table on the left twice: **"From the LEFT side, which rows have no partner on the right?"** — the preserved/returned side is always the left one, so the subject of the question goes on the left.

In [ ]:
%sql
-- SQL version: full outer + null filter (works, but the roundabout way)
select c.customer_id, c.customer_name, c.city
from orders_p16 as o
full outer join customers_p16 as c
  on c.customer_id = o.customer_id
where o.order_id is null

-- 4 customers have never ordered: CU-104, CU-110, CU-116, CU-121

In [ ]:
# API version: the direct pattern
never_ordered_customers_df = (
    customers_clean_p16_df
    .join(orders_clean_p16_df, on="customer_id", how="left_anti")
)
never_ordered_customers_df.display()   # same 4 customers -> cross-validated

# The reverse direction answers the other half of the question:
# orders whose customer is NOT in the CRM (the 2 orphans)
orphan_orders_df = (
    orders_clean_p16_df
    .join(customers_clean_p16_df, on="customer_id", how="left_anti")
)
orphan_orders_df.display()   # ORD-5108 (CU-888), ORD-5109 (CU-999) -> kept + flagged in silver

**Answer:** 4 customers never ordered (CU-104 Daan Visser, CU-110 Joris Hendriks, CU-116 Pien Veenstra, CU-121 Umut Maas). Before the key-casing fix this query returned 6 "unknowns" on the orphan side — the anti join was the tool that exposed the dirty keys.

### G2 [W] — Monthly revenue + month-over-month change

Two lessons happened live in this question:

1. **Multiply at row level, then sum.** My first version computed `sum(quantity) * sum(unit_price)` — that multiplies every quantity with every price and massively inflates revenue. Correct: `sum(quantity * unit_price)`.
2. **Writing SQL and the API is cross-validation, not just practice.** My first API version quietly forgot the `delivered` filter — every month came out too high. Comparing the two outputs caught it immediately.

In [ ]:
%sql
with monthly_revenue as (
  select
    sum(quantity)              as total_quantity,
    sum(quantity * unit_price) as total_revenue,
    date_format(order_date, "yyyy-MM") as order_month
  from orders_p16
  where status = "delivered"
  group by order_month
)
select
  order_month,
  total_quantity,
  total_revenue,
  lag(total_revenue) over (order by order_month)          as previous_revenue,
  total_revenue - lag(total_revenue) over (order by order_month) as revenue_diff
from monthly_revenue
order by order_month

In [ ]:
from pyspark.sql.functions import date_format, sum as sum_, lag, expr
from pyspark.sql.window import Window

monthly_revenue_p16_df = (
    orders_clean_p16_df
    .withColumn("order_month", date_format("order_date", "yyyy-MM"))
    .filter(col("status") == "delivered")            # filter BEFORE groupBy: after agg the column is gone
    .groupBy("order_month")
    .agg(
        sum_("quantity").alias("total_quantity"),
        sum_(col("quantity") * col("unit_price")).alias("total_revenue")
    )
)

w = Window.orderBy("order_month")   # global window on purpose: months are one sequence (6 rows, no partition needed)

(
    monthly_revenue_p16_df
    .withColumn("previous_revenue", lag("total_revenue").over(w))
    .withColumn("revenue_diff", expr("total_revenue - previous_revenue"))
    .orderBy("order_month")          # groupBy output order is NOT guaranteed - sort before reading
    .display()
)

**Answer (delivered orders, EUR):**

| month | revenue | vs previous |
|---|---|---|
| 2025-01 | 253.70 | null (no previous month) |
| 2025-02 | 390.25 | +136.55 |
| 2025-03 | 394.55 | +4.30 |
| 2025-04 | **565.30** | +170.75 ← strongest month |
| 2025-05 | 354.10 | −211.20 |
| 2025-06 | 435.60 | +81.50 |

SQL and API agree — cross-validated.

### G3 [W] — Orders above their category's average order value

The result must stay at **one row per order**. That is exactly what `groupBy` cannot do: grouping by category would collapse 73 delivered orders into 4 category rows, and the per-order comparison would be impossible. A **window average partitioned by category** attaches the category average to every order row without changing the grain — this needs the `books` join from silver (category lives there).

In [ ]:
from pyspark.sql.functions import avg

w_cat = Window.partitionBy("category")   # no orderBy: we want the whole-category average on every row,
                                         # not a running average

g3_df = (
    silver_p16_df
    .filter(col("status") == "delivered")
    .withColumn("order_value", col("quantity") * col("unit_price"))
    .withColumn("category_avg_order_value", avg("order_value").over(w_cat))
    .withColumn("above_category_avg", col("order_value") > col("category_avg_order_value"))
)

g3_df.select("order_id", "category", "order_value",
             "category_avg_order_value", "above_category_avg").display()

print(g3_df.filter(col("above_category_avg")).count())   # 40 orders above their category average
g3_df.filter(col("above_category_avg")).groupBy("category").count().display()

**Answer:** **40** delivered orders sit above their category's average order value (children 11 of 21, cookbooks 9 of 19, fiction 9 of 15, non-fiction 11 of 18). Grain of the result: still one row per order — the window added context columns without collapsing anything. Orders with a null price drop out naturally: `null > avg` is null, which the filter treats as false.

## 6. Idempotent write + final checks

`overwrite` makes the write safe to re-run: running the notebook twice produces the same table, not doubled data.

In [ ]:
(
    silver_p16_df.write
    .mode("overwrite")
    .saveAsTable("bookstore_orders_silver")
)

# Final checks - the numbers I would defend in the meeting
final_df = spark.table("bookstore_orders_silver")
print(final_df.count())                                          # 109 = expected row count
print(final_df.select("order_id").distinct().count())            # 109 = grain still holds
print(final_df.filter(col("customer_missing_in_crm")).count())   # 2 flagged orphans

# Reconciliation: silver's delivered revenue must equal the G2 monthly total
from pyspark.sql.functions import round as round_
(
    final_df.filter(col("status") == "delivered")
    .select(round_(sum_(col("quantity") * col("unit_price")), 2).alias("delivered_revenue"))
    .display()                                                   # 2393.50 = sum of the G2 months
)

## 7. Bonus — unit tests on the G1 logic

The anti-join logic moved into a plain function (`bookstore_functions.py`), following the three extraction rules: it **returns** a DataFrame, takes DataFrames as **parameters**, and knows nothing about file paths.

```python
from pyspark.sql.functions import expr

def customers_without_orders(customers_df, orders_df):
    df = (
        customers_df.alias("c")
        .join(orders_df.alias("o"), on=expr("c.customer_id = o.customer_id"), how="left_anti")
    )
    return df.select("customer_id", "customer_name")
```

Two tests in `test_bookstore_functions.py`, both with tiny hand-made data built via `createDataFrame`:

1. **Happy path** — 4 customers, orders for C1 and C3 only → expected written **by hand** (C2 Bram, C4 Daan). No circular proof: the expected rows never come from the function itself. (My first draft had the expected list inverted — the customers *with* orders. Writing the expected on paper first is the whole point.)
2. **Empty case** — every customer has an order → the function must return **0 rows**, not invent any. One test proves one behaviour: the happy path shows it finds what it should; the empty case shows it doesn't fabricate what it shouldn't — a wrong join type (`left` instead of `left_anti`) would pass nowhere here.

In [ ]:
import pytest
import sys
sys.dont_write_bytecode = True

retcode = pytest.main(["-v", "test_bookstore_functions.py"])
assert retcode == 0    # 2 passed - and the notebook fails loudly if the tests ever break

## 8. Decision log

| # | decision | evidence / number | alternative cost |
|---|---|---|---|
| 1 | Dropped 3 full-row duplicates | 112 → 109; ORD-5073/5085/5097 identical in every column | keeping them double-counts ~3 orders of revenue |
| 2 | Normalized `customer_id` with trim + **upper** on the orders side | anti join went from 6 "unknown" customers to 2 real ones | without it, 4 real customers' orders silently fail every join |
| 3 | Kept the 2 orphan orders via **left join + flag** | EUR 56.85 of delivered revenue (ORD-5108, ORD-5109) | inner join silently understates revenue; upstream loses the DQ signal |
| 4 | 3 unparseable prices and 2 unknown statuses → NULL, rows kept | placeholders measured before and after the cast (3 = 3) | deleting the rows loses valid quantity/date information |
| 5 | Revenue = `sum(quantity * unit_price)` at row level | first draft (`sum(qty) * sum(price)`) gave 2355.15 for January vs the correct 253.70 | sums-multiplied inflates revenue ~9x |
| 6 | Money as `decimal(10,2)`, dates via 3-format `coalesce(try_to_date)` | 0 unparsed dates; post-cast null count unchanged | int cast truncates cents; single-format parse nulls 40% of dates |

## 9. Defence questions & answers

**1. You normalized the customer IDs before joining. What would have happened if you had skipped it? Would Spark have raised an error?**

Spark would not raise any error — string join keys just wouldn't match, so the rows would silently fall out of the join. I saw this live: my anti join first returned 6 "unknown" customers, but 4 of them were just lowercase variants of real IDs. After normalizing with trim and upper, only 2 real orphans remained. That's why I treat silent mismatches as more dangerous than errors — an error stops you, a silent mismatch gives you wrong numbers you might report.

**2. For the orders whose customer_id does not exist in the CRM: inner or left join, and why? What does each choice do to revenue?**

I used a left join with orders on the left, because those 2 orders are real sales — real money (EUR 56.85). An inner join would silently drop them and my revenue report would be understated. With the left join the revenue stays complete; the customer columns are just null for those 2 rows, and I flag them so the CRM mismatch is visible upstream instead of hidden by a join choice.

**3. In G3, why does a window function work where `groupBy` would not? What is the grain of your G3 result?**

`groupBy("category")` collapses the data to 4 category rows — the individual orders are gone, so there is nothing left to compare against the average. A window average partitioned by category attaches the category average to **every order row** without collapsing anything. The grain of my G3 result is still one row per delivered order; the window only added context columns. And because I want the whole-category average, the window has **no orderBy** — adding one would silently turn it into a running average.

**4. How can you prove your join did not duplicate any order rows?**

I compare the row count before and after the join: 109 orders before, and it must still be 109 after — a join should add columns, not rows. If the count goes up, that's fan-out and it means the join key is not unique on the other side. That's also why I checked during profiling that `customer_id` is unique in the customers table (22 rows, 22 distinct): that uniqueness is what *guarantees* no fan-out, and the count check is the proof.

## Key takeaways

1. **Dirty join keys fail silently.** No error, no warning — rows just stop matching. Normalize keys to one canonical format on both sides before any join, and let an anti join expose what still doesn't match.
2. **The returned side of a left/anti join is the LEFT side.** Put the subject of the question on the left. ("From the left side, which rows have no partner on the right?")
3. **Join type is a business decision.** Inner vs left decides whether unmatched money silently disappears from a revenue report. Keep + flag beats drop + hide.
4. **Multiply at row level, then aggregate.** `sum(a*b)` ≠ `sum(a) * sum(b)` — the second inflated January revenue ~9x before I caught it.
5. **Writing SQL and the DataFrame API for the same question is cross-validation.** The API draft was missing a filter; comparing outputs caught it in seconds.
6. **A green test run proves nothing until you read WHAT ran.** My first pytest run passed — on the previous project's test file. Read the collected test names, and break a test on purpose once to see real red.
7. **One test proves one behaviour.** Happy path + empty case cover different failure modes; hand-written expected rows are the only honest oracle.